# Part 1: Decoupled Parameter Threshold Simulation

In [ ]:
# 1. Install required quantum libraries
!pip install stim sinter pymatching pandas matplotlib

import stim
import sinter
import pandas as pd
import matplotlib.pyplot as plt

# 2. Setup the experiment parameters
distances = [3, 5, 7]
measurement_sweep = [0.01, 0.02, 0.04, 0.06, 0.08, 0.10, 0.13, 0.16]
gate_scenarios = {'Clean_Gates_0.1pc': 0.001, 'Moderate_Gates_0.5pc': 0.005, 'Noisy_Gates_1.0pc': 0.010}

all_tasks = []
for scenario_name, fixed_gate_p in gate_scenarios.items():
    for d in distances:
        for m_p in measurement_sweep:
            circuit = stim.Circuit.generated(
                code_task="surface_code:rotated_memory_z",
                distance=d,
                rounds=d,
                after_clifford_depolarization=fixed_gate_p,
                after_reset_flip_probability=m_p
            )
            all_tasks.append(sinter.Task(
                circuit=circuit,
                json_metadata={'scenario': scenario_name, 'd': d, 'gate_p': fixed_gate_p, 'meas_p': m_p}
            ))

print("Step 1: Running simulations across 3 gate noise settings (10,000 shots each)...")
stats = sinter.collect(
    num_workers=4, max_shots=10000, max_errors=300, tasks=all_tasks, decoders=['pymatching']
)

# 3. Compile and save the complete dataset to CSV
parsed_rows = []
for s in stats:
    parsed_rows.append({
        'Scenario': s.json_metadata['scenario'],
        'Distance': s.json_metadata['d'],
        'Gate_Noise': s.json_metadata['gate_p'],
        'Measurement_Noise': s.json_metadata['meas_p'],
        'Shots': s.shots,
        'Logical_Errors': s.errors,
        'Logical_Error_Rate': s.errors / s.shots
    })

df = pd.DataFrame(parsed_rows)
df.to_csv('advanced_decoupled_phase_data.csv', index=False)
print("SUCCESS: 'advanced_decoupled_phase_data.csv' saved to your workspace side folder!")

# 4. FIXED GRAPH GENERATION: Loop over all scenarios to draw separate charts!
colors = {3: 'royalblue', 5: 'crimson', 7: 'forestgreen'}

for scenario_name, fixed_gate_p in gate_scenarios.items():
    plt.figure(figsize=(7, 5))
    scenario_df = df[df['Gate_Noise'] == fixed_gate_p]
    for d_val in distances:
        plot_df = scenario_df[scenario_df['Distance'] == d_val].sort_values(by='Measurement_Noise')
        plt.plot(plot_df['Measurement_Noise'], plot_df['Logical_Error_Rate'], marker='o',
                 color=colors[d_val], linewidth=2.5, label=f"Distance (d) = {d_val}")
    plt.title(f"2D Surface Code: Decoupled Parameter Threshold\n(Gate Noise Locked at {fixed_gate_p * 100}%)", fontsize=11, fontweight='bold')
    plt.xlabel("Physical Measurement Error Rate (Readout Noise)", fontsize=10)
    plt.ylabel("Logical Error Rate P_L (Log Scale)", fontsize=10)
    plt.yscale('log')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend(loc="lower right")
    filename = f"threshold_plot_{scenario_name}.png"
    plt.savefig(filename, bbox_inches='tight', dpi=150)
    print(f"SUCCESS: Saved visual asset '{filename}' to your side panel folder!")
    plt.show()


# Part 2: Syndrome Repetition Optimization (Google Hardware - Flat Curves)

In [ ]:

# GOOGLE HARDWARE PROFILE: We lock the asymmetric noise channels as fixed baselines
NATIVE_GATE_NOISE = 0.006     # Locked at Google's 0.6% CZ gate error rate
NATIVE_READOUT_NOISE = 0.020  # Locked at Google's 2.0% measurement readout floor

experiment_grid = [3, 5, 7]
simulation_jobs = []

for code_d in experiment_grid:
    max_cycles = (2 * code_d) + 1
    for current_T in range(1, max_cycles):
        layout = stim.Circuit.generated(
            code_task="surface_code:rotated_memory_z",
            distance=code_d,
            rounds=current_T,
            after_clifford_depolarization=NATIVE_GATE_NOISE,
            after_reset_flip_probability=NATIVE_READOUT_NOISE
        )
        simulation_jobs.append(sinter.Task(
            circuit=layout,
            json_metadata={'size_d': code_d, 'rounds_T': current_T}
        ))

print("Executing automated Monte Carlo loops (15,000 runs per point)...")
raw_data = sinter.collect(
    num_workers=4, max_shots=15000, max_errors=400, tasks=simulation_jobs, decoders=['pymatching']
)

data_log = []
for entry in raw_data:
    data_log.append({
        'Grid_Distance': entry.json_metadata['size_d'],
        'Readout_Ticks': entry.json_metadata['rounds_T'],
        'Net_Failure_Rate': entry.errors / entry.shots
    })

results_frame = pd.DataFrame(data_log)
results_frame.to_csv('syndrome_optimization_results.csv', index=False)
print("Data table saved locally as 'syndrome_optimization_results.csv'")

plt.figure(figsize=(8, 6))
palette = {3: 'royalblue', 5: 'crimson', 7: 'forestgreen'}

print("\n--- CALCULATED OPTIMAL VALUES FOUND ---")
for d_val in experiment_grid:
    subset = results_frame[results_frame['Grid_Distance'] == d_val].sort_values(by='Readout_Ticks')
    x_axis = subset['Readout_Ticks']
    y_axis = subset['Net_Failure_Rate']
    winning_index = y_axis.idxmin()
    optimal_T = subset.loc[winning_index, 'Readout_Ticks']
    lowest_error = subset.loc[winning_index, 'Net_Failure_Rate']
    print(f"Lattice Size d={d_val} -> Target Sweet Spot T* = {optimal_T} Rounds (Error Floor: {lowest_error*100:.3f}%)")
    plt.plot(x_axis, y_axis, marker='o', color=palette[d_val], linewidth=2.5, label=f"Distance (d) = {d_val}")
    plt.axvline(x=optimal_T, color=palette[d_val], linestyle='--', alpha=0.5)

plt.title("Syndrome Repetition Profiles (Google Hardware Asymmetry)", fontsize=11, fontweight='bold')
plt.xlabel("Number of Syndrome Measurement Rounds (T)", fontsize=10)
plt.ylabel("Net Logical Error Rate P_L (Log Scale)", fontsize=10)
plt.yscale('log')
plt.xticks(range(1, 15))
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.savefig("repetition_optimization_flat_curves.png", bbox_inches='tight', dpi=150)
plt.show()


# Part 3: Syndrome Repetition Optimization (0.1% Gate Noise Baseline - U-Shaped Valley Curves)

In [ ]:

# DYNAMIC SWEEP WITH OPTIMIZED GATE NOISE (TO REVEAL THE U-SHAPE)
CLEAN_GATE_NOISE = 0.001
SLOPPY_READOUT_NOISE = 0.060

experiment_grid = [3, 5, 7]
new_jobs = []

for code_d in experiment_grid:
    max_cycles = (2 * code_d) + 1
    for current_T in range(1, max_cycles):
        layout = stim.Circuit.generated(
            code_task="surface_code:rotated_memory_z",
            distance=code_d,
            rounds=current_T,
            after_clifford_depolarization=CLEAN_GATE_NOISE,
            after_reset_flip_probability=SLOPPY_READOUT_NOISE
        )
        new_jobs.append(sinter.Task(
            circuit=layout,
            json_metadata={'size_d': code_d, 'rounds_T': current_T}
        ))

print("Running validation loop (15,000 shots per point)...")
optimized_data = sinter.collect(
    num_workers=4, max_shots=15000, max_errors=400, tasks=new_jobs, decoders=['pymatching']
)

data_log = []
for entry in optimized_data:
    data_log.append({
        'Grid_Distance': entry.json_metadata['size_d'],
        'Readout_Ticks': entry.json_metadata['rounds_T'],
        'Net_Failure_Rate': entry.errors / entry.shots
    })

df_valley = pd.DataFrame(data_log)
plt.figure(figsize=(8, 6))
palette = {3: 'royalblue', 5: 'crimson', 7: 'forestgreen'}

for d_val in experiment_grid:
    subset = df_valley[df_valley['Grid_Distance'] == d_val].sort_values(by='Readout_Ticks')
    x = subset['Readout_Ticks']
    y = subset['Net_Failure_Rate']
    t_star = subset.loc[y.idxmin(), 'Readout_Ticks']
    plt.plot(x, y, marker='o', color=palette[d_val], linewidth=2.5, label=f"Distance (d) = {d_val}")
    plt.axvline(x=t_star, color=palette[d_val], linestyle='--', alpha=0.5)

plt.title("Syndrome Repetition Optimization Profiles (U-Shaped Valley)", fontsize=11, fontweight='bold')
plt.xlabel("Number of Syndrome Measurement Rounds (T)", fontsize=10)
plt.ylabel("Net Logical Error Rate P_L (Log Scale)", fontsize=10)
plt.yscale('log')
plt.xticks(range(1, 15))
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.savefig("repetition_optimization_valley.png", bbox_inches='tight', dpi=150)
plt.show()
